### 1. Import libraries and load data from database.

In [1]:
# import libraries
import nltk
nltk.download(['punkt', 'wordnet'])

import os
import re
import numpy as np
import pandas as pd
import sqlite3
from sqlalchemy import create_engine,inspect

from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, HashingVectorizer
from sklearn.metrics import hamming_loss, confusion_matrix, classification_report, precision_recall_fscore_support, accuracy_score, precision_score, recall_score, f1_score

c:\Users\sinde\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# Move to datasets folder
original_directory = os.getcwd()
dataset_directory = './dataset'
os.chdir(dataset_directory)

In [3]:
#Look for the tables name in the SQL database
engine = create_engine('sqlite:///DisasterResponse.db')

# Create an inspector
inspector = inspect(engine)

# Get the list of table names
table_names = inspector.get_table_names()

table_names

['messages']

In [4]:
# Import data
engine = create_engine('sqlite:///DisasterResponse.db')
with engine.connect() as connection:
    df = pd.read_sql("SELECT * FROM messages", connection)
engine.dispose()

In [14]:
# get target and feature data - first part of ML
df_r = df[['message', 'related']]
df_related_1 = df_r[df_r['related'] == 1]
df_related_0 = df_r[df_r['related'] == 0]

# Set the random seed for reproducibility
random_seed = 42

# Resample to balance the classes
df_related_1 = df_related_1.sample(len(df_related_0), replace=True, random_state=random_seed)
df_balanced = pd.concat([df_related_0, df_related_1]).sample(frac=1, random_state=random_seed).reset_index(drop=True)

# New datasets
X = df_balanced['message']
Y = df_balanced['related']

In [15]:
X.head()

0    - To strengthen existing livelihood activities...
1    hello, i would like to leave for the us, just ...
2    NOTES: This kind of message is not important f...
3    The ninth municipal section Citroniera in Leog...
4    Since the Oct. 8 earthquake, UNHCR has airlift...
Name: message, dtype: object

In [16]:
Y.value_counts()

1    6122
0    6122
Name: related, dtype: int64

### tokenization function to process your text data

In [106]:
def tokenize(text):
    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()

    clean_tokens = []
    for tok in tokens:
        clean_tok = lemmatizer.lemmatize(tok).lower().strip()
        clean_tokens.append(clean_tok)

    return clean_tokens

### Testing mutiple models

In [124]:
# Split the data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=random_seed)

In [141]:
pipeline = Pipeline([
    ('vect', CountVectorizer(tokenizer=custom_tokenizer, ngram_range=(1,3))),
    ('tfidf', TfidfTransformer()),
    ('clf', RandomForestClassifier())
])

param_grid = [
    {
        'clf': [RandomForestClassifier()],
        'clf__n_estimators': [100, 200],
        'clf__min_samples_split': [2, 5]
    },
    {
        'clf': [LogisticRegression(max_iter=1000)],
        'clf__C': [0.1, 1, 10],
        'clf__solver': ['liblinear', 'saga']
    },
    {
        'clf': [GradientBoostingClassifier()],
        'clf__n_estimators': [100, 200],
        'clf__learning_rate': [0.01, 0.1, 0.2]
    }
]

In [142]:
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='precision_weighted', n_jobs=1, verbose=2)
grid_search.fit(X_train, Y_train)

# Print the best parameters and best score
print(f'Best parameters found: {grid_search.best_params_}')
print(f'Best Precision score: {grid_search.best_score_}')

Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  12.5s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  12.4s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  12.2s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time=  22.7s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time=  22.9s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time=  23.3s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=100; total time=   9.5s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=100; total time=  10.7s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=

In [144]:
# Evaluate the model
precision = precision_score(Y_test, Y_pred, average='weighted', zero_division=0)
recall = recall_score(Y_test, Y_pred, average='weighted', zero_division=0)
f1 = f1_score(Y_test, Y_pred, average='weighted', zero_division=0)
overall_accuracy = accuracy_score(Y_test, Y_pred)

print(f'Overall Accuracy: {overall_accuracy:.4f}')
print(f'Macro Average Precision: {precision:.4f}')
print(f'Macro Average Recall: {recall:.4f}')
print(f'Macro Average F1 Score: {f1:.4f}')

Overall Accuracy: 0.7972
Macro Average Precision: 0.7994
Macro Average Recall: 0.7972
Macro Average F1 Score: 0.7969
